# Experimentos 4 e 5 v2 — CoT + Instruction Tuning (Somente Modelos Locais)

**Experimento 4 — Chain of Thought (CoT) com Optuna:**
- Llama 3.1 8B, Mistral 7B, Qwen 2.5 7B, Gemma 2 9B
- Optuna otimiza temperatura e max_chars
- Gera feedbacks textuais que alimentam o Exp5

**Experimento 5 — Instruction Tuning com Optuna:**
- Mesmos 4 modelos, treinados com os feedbacks do Exp4
- Optuna otimiza lr, lora_rank, batch, epochs

**Sem uso de APIs externas — somente GPU local.**

**Secrets necessarios:** HF_TOKEN

Ambiente de execucao -> Alterar tipo -> GPU A100

In [ ]:
# Celula 1 - Instalacao
!pip install -q transformers accelerate bitsandbytes gdown scikit-learn matplotlib peft trl optuna datasets
print('Instalado!')

In [ ]:
# Celula 2 - Imports
import re, gc, json, time, random, os
import numpy as np
import pandas as pd
import torch
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    cohen_kappa_score, f1_score
)
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
from datasets import Dataset
from google.colab import userdata
from huggingface_hub import login, HfApi

optuna.logging.set_verbosity(optuna.logging.WARNING)

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

print('Autenticado!')

In [ ]:
# Celula 3 - Dataset e K-Fold
import gdown

gdown.download(
    'https://drive.google.com/uc?id=1zg9n7EUDWKfu6t_f3aYDS_QroYXGNGwd',
    'meu_dataset.csv', quiet=False
)
df_enem = pd.read_csv('meu_dataset.csv')


def limpar(texto):
    if pd.isna(texto): return ''
    texto = str(texto).strip("[]'\" ")
    texto = texto.replace('\n', ' ')
    texto = re.sub(r'\[[A-Z/]+\]', '', texto)
    texto = re.sub(r'\{[a-z]+\}', '', texto)
    return re.sub(r'\s+', ' ', texto).strip()


df_enem['essay_limpo'] = df_enem['essay'].apply(limpar)
df_enem = df_enem[df_enem['score'] > 0].reset_index(drop=True)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
df_enem['fold'] = -1
for i, (tr, te) in enumerate(skf.split(df_enem, df_enem['score'])):
    df_enem.loc[te, 'fold'] = i

df_treino = df_enem[df_enem['fold'] != 0].reset_index(drop=True)
df_teste  = df_enem[df_enem['fold'] == 0].reset_index(drop=True)
print(f'Total: {len(df_enem)} | Treino: {len(df_treino)} | Teste: {len(df_teste)}')

In [ ]:
# Celula 4 - Funcoes auxiliares

def extrair_notas(resposta):
    try:
        texto = re.sub(r'```json|```', '', str(resposta))
        i = texto.find('{')
        j = texto.rfind('}') + 1
        if i == -1 or j <= i: return None
        dados = json.loads(texto[i:j])
        comps = ['C1', 'C2', 'C3', 'C4', 'C5']
        if all(c in dados for c in comps):
            if max(dados[c] for c in comps) <= 20:
                for c in comps: dados[c] *= 10
            dados['Nota_Total'] = sum(dados[c] for c in comps)
            return dados
        if 'Nota_Total' in dados: return dados
    except: pass
    return None


def extrair_notas_markdown(resposta):
    try:
        texto = str(resposta)
        comps = {}
        for c in ['C1', 'C2', 'C3', 'C4', 'C5']:
            m = re.search(rf'{c}[^:]*:\s*(\d+)', texto)
            if m: comps[c] = int(m.group(1))
        if len(comps) == 5:
            if max(comps.values()) <= 20:
                for c in comps: comps[c] *= 10
            comps['Nota_Total'] = sum(comps.values())
            return comps
        m = re.search(r'(?:Total|Nota\s*Total|Nota\s*Final)[^\d]*(\d{3,4})', texto, re.I)
        if m: return {'Nota_Total': int(m.group(1))}
    except: pass
    return None


def extrair(resposta):
    return extrair_notas(resposta) or extrair_notas_markdown(resposta)


def extrair_feedback(resposta):
    try:
        t = str(resposta)
        j = t.rfind('}')
        fb = t[j + 1:].strip() if j >= 0 else ''
        if len(fb) < 20:
            inicio = t.find('{')
            fb = t[:inicio].strip() if inicio >= 0 else ''
        return fb if len(fb) > 20 else 'Feedback nao disponivel'
    except:
        return 'Feedback nao disponivel'


def calcular_metricas(y_true, y_pred, nome):
    yt = np.array(y_true, dtype=float)
    yp = np.array(y_pred, dtype=float)
    mae  = mean_absolute_error(yt, yp)
    rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    def disc(n): return np.clip(np.round(np.array(n) / 40).astype(int), 0, 25)
    try:    qwk = cohen_kappa_score(disc(yt), disc(yp), weights='quadratic')
    except: qwk = float('nan')
    try:    f1  = f1_score(disc(yt), disc(yp), average='weighted', zero_division=0)
    except: f1  = float('nan')
    print(f'\n{"="*55}')
    print(f'METRICAS — {nome}')
    print(f'{"="*55}')
    print(f'  Amostras : {len(yt)}')
    print(f'  MAE      : {mae:.4f}')
    print(f'  RMSE     : {rmse:.4f}')
    print(f'  QWK      : {qwk:.4f}')
    print(f'  F1 Score : {f1:.4f}')
    print(f'{"="*55}')
    return {'modelo': nome, 'mae': mae, 'rmse': rmse, 'qwk': qwk, 'f1': f1, 'n': len(yt)}


print('Funcoes auxiliares carregadas!')

In [ ]:
# Celula 5 - Inferencia e modelos

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16
)


def carregar_modelo(nome_modelo, para_treino=False):
    print(f'Carregando {nome_modelo}...')
    tok = AutoTokenizer.from_pretrained(nome_modelo, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    if para_treino: tok.padding_side = 'right'
    m = AutoModelForCausalLM.from_pretrained(
        nome_modelo, quantization_config=bnb_config,
        device_map='auto', trust_remote_code=True
    )
    if para_treino:
        m.config.use_cache = False
    else:
        m.eval()
    print('Carregado!')
    return tok, m


def liberar(m, tok):
    del m, tok
    gc.collect()
    torch.cuda.empty_cache()
    print('Memoria GPU liberada.')


def inf_local_cot(tok, m, redacao, temp=0.2, max_tok=700):
    prompt = (
        'Voce e um avaliador oficial de redacoes do ENEM.\n'
        'Avalie a redacao seguindo a escala: 0, 40, 80, 120, 160 ou 200 pontos por competencia.\n\n'
        'COMPETENCIAS:\n'
        '- C1 (Norma Culta): dominio da norma padrao (0-200)\n'
        '- C2 (Tema/Estrutura): adequacao ao tema e estrutura (0-200)\n'
        '- C3 (Argumentacao): selecao e organizacao de argumentos (0-200)\n'
        '- C4 (Coesao): uso de mecanismos linguisticos (0-200)\n'
        '- C5 (Proposta de Intervencao): proposta com agente, acao, meio, efeito (0-200)\n\n'
        'REDACAO:\n' + redacao + '\n\n'
        'Siga EXATAMENTE estas etapas:\n\n'
        'PASSO 1 - ANALISE:\nAnalise cada competencia separadamente.\n\n'
        'PASSO 2 - NOTAS:\nAtribua as notas no formato JSON:\n'
        '{"C1": valor, "C2": valor, "C3": valor, "C4": valor, "C5": valor, "Nota_Total": soma}\n\n'
        'PASSO 3 - FEEDBACK:\nEscreva 2-3 paragrafos de feedback construtivo.'
    )
    msgs = [{'role': 'user', 'content': prompt}]
    try:
        txt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except:
        txt = prompt
    inp = tok(txt, return_tensors='pt', truncation=True, max_length=3072).to(m.device)
    with torch.no_grad():
        out = m.generate(
            **inp, max_new_tokens=max_tok,
            temperature=temp, do_sample=True,
            pad_token_id=tok.pad_token_id
        )
    return tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()


def inf_local_it(tok, m, redacao, temp=0.2, max_tok=600):
    prompt = (
        'Voce e um avaliador oficial de redacoes do ENEM.\n'
        'Avalie a redacao fornecendo notas e feedback construtivo.\n'
        'Escala: 0, 40, 80, 120, 160 ou 200 por competencia.\n'
        'C1(Norma Culta) C2(Tema/Estrutura) C3(Argumentacao) C4(Coesao) C5(Intervencao)\n\n'
        'REDACAO:\n' + redacao + '\n\nNOTAS e FEEDBACK:'
    )
    msgs = [{'role': 'user', 'content': prompt}]
    try:
        txt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except:
        txt = prompt
    inp = tok(txt, return_tensors='pt', truncation=True, max_length=3072).to(m.device)
    with torch.no_grad():
        out = m.generate(
            **inp, max_new_tokens=max_tok,
            temperature=temp, do_sample=True,
            pad_token_id=tok.pad_token_id
        )
    return tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()


print('Funcoes de inferencia carregadas!')

# EXPERIMENTO 4 — Chain of Thought (CoT) com Optuna

Cada modelo local e otimizado pelo Optuna (temperatura e max_chars) e depois
avaliado em todo o fold 0. Os feedbacks gerados sao salvos nos CSVs ck4_*.csv
e usados como gabarito no Experimento 5.

In [ ]:
# Celula 6 - Optuna e runner CoT

def obj_optuna_cot(trial, fn_inf, n_val=10):
    temp      = trial.suggest_float('temp', 0.05, 0.5)
    max_chars = trial.suggest_int('max_chars', 500, 2000, step=250)

    df_val = df_enem[df_enem['fold'] == 1].head(n_val)
    yp, yt = [], []
    for _, row in df_val.iterrows():
        try:
            resp  = fn_inf(row['essay_limpo'][:max_chars], temp)
            notas = extrair(resp)
            if notas and 'Nota_Total' in notas:
                yp.append(notas['Nota_Total'])
                yt.append(row['score'])
        except: pass
    if len(yp) < 5: raise optuna.TrialPruned()
    def disc(n): return np.clip(np.round(np.array(n) / 40).astype(int), 0, 25)
    try:    return cohen_kappa_score(disc(yt), disc(yp), weights='quadratic')
    except: return -1.0


def rodar_optuna_cot(fn_inf, nome, n_trials=8):
    print(f'Optuna CoT: otimizando {nome} ({n_trials} trials)...')
    study = optuna.create_study(
        direction='maximize',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5)
    )
    study.optimize(lambda t: obj_optuna_cot(t, fn_inf), n_trials=n_trials, catch=(Exception,))
    print(f'Melhores params: {study.best_params}')
    return study.best_params


def rodar_cot(nome, fn_inf, params, ckpt_csv):
    temp      = params.get('temp', 0.2)
    max_chars = params.get('max_chars', None)

    if os.path.exists(ckpt_csv):
        df_ck     = pd.read_csv(ckpt_csv)
        ja_feitos = set(df_ck['index_redacao'].tolist())
        print(f'Checkpoint: {len(ja_feitos)} ja processadas')
    else:
        df_ck     = pd.DataFrame(columns=['index_redacao', 'score', 'pred_total', 'feedback'])
        ja_feitos = set()

    pendentes = df_teste[~df_teste.index.isin(ja_feitos)]
    print(f'Pendentes: {len(pendentes)}/{len(df_teste)}')

    novos = []
    for idx, (i, row) in enumerate(pendentes.iterrows()):
        try:
            redacao = row['essay_limpo'][:max_chars] if max_chars else row['essay_limpo']
            resp    = fn_inf(redacao, temp)
            notas   = extrair(resp)
            pred    = notas['Nota_Total'] if notas else None
            fb      = extrair_feedback(resp)
            novos.append({'index_redacao': i, 'score': row['score'], 'pred_total': pred, 'feedback': fb})
        except:
            novos.append({'index_redacao': i, 'score': row['score'], 'pred_total': None, 'feedback': None})

        if (idx + 1) % 50 == 0:
            df_ck = pd.concat([df_ck, pd.DataFrame(novos)], ignore_index=True)
            df_ck.to_csv(ckpt_csv, index=False)
            novos = []
            print(f'  Checkpoint: {idx + 1 + len(ja_feitos)}/{len(df_teste)}')

    if novos:
        df_ck = pd.concat([df_ck, pd.DataFrame(novos)], ignore_index=True)
        df_ck.to_csv(ckpt_csv, index=False)

    df_v = df_ck.dropna(subset=['pred_total'])
    print(f'Validas: {len(df_v)}/{len(df_teste)}')
    if len(df_v) > 0:
        return calcular_metricas(df_v['score'].tolist(), df_v['pred_total'].tolist(), nome)
    return None


print('Optuna CoT e runner carregados!')

## Llama 3.1 8B — CoT

In [ ]:
tok, m = carregar_modelo('meta-llama/Llama-3.1-8B-Instruct')
fn_llama_cot = lambda r, t: inf_local_cot(tok, m, r, temp=t)
params_llama_cot = rodar_optuna_cot(fn_llama_cot, 'Llama 3.1 8B', n_trials=8)
res_llama_cot = rodar_cot('Llama 3.1 8B (CoT)', fn_llama_cot, params_llama_cot, 'ck4_llama.csv')
liberar(m, tok)

## Mistral 7B — CoT

In [ ]:
tok, m = carregar_modelo('mistralai/Mistral-7B-Instruct-v0.3')
fn_mistral_cot = lambda r, t: inf_local_cot(tok, m, r, temp=t)
params_mistral_cot = rodar_optuna_cot(fn_mistral_cot, 'Mistral 7B', n_trials=8)
res_mistral_cot = rodar_cot('Mistral 7B (CoT)', fn_mistral_cot, params_mistral_cot, 'ck4_mistral.csv')
liberar(m, tok)

## Qwen 2.5 7B — CoT

In [ ]:
tok, m = carregar_modelo('Qwen/Qwen2.5-7B-Instruct')
fn_qwen_cot = lambda r, t: inf_local_cot(tok, m, r, temp=t)
params_qwen_cot = rodar_optuna_cot(fn_qwen_cot, 'Qwen 2.5 7B', n_trials=8)
res_qwen_cot = rodar_cot('Qwen 2.5 7B (CoT)', fn_qwen_cot, params_qwen_cot, 'ck4_qwen.csv')
liberar(m, tok)

## Gemma 2 9B — CoT

In [ ]:
tok, m = carregar_modelo('google/gemma-2-9b-it')
fn_gemma_cot = lambda r, t: inf_local_cot(tok, m, r, temp=t)
params_gemma_cot = rodar_optuna_cot(fn_gemma_cot, 'Gemma 2 9B', n_trials=8)
res_gemma_cot = rodar_cot('Gemma 2 9B (CoT)', fn_gemma_cot, params_gemma_cot, 'ck4_gemma.csv')
liberar(m, tok)

## Resultados Exp4

In [ ]:
# Celula - Resultados Exp4
import matplotlib.pyplot as plt

todos_cot = [res_llama_cot, res_mistral_cot, res_qwen_cot, res_gemma_cot]
df_cot = pd.DataFrame([r for r in todos_cot if r is not None])
df_cot = df_cot.sort_values('qwk', ascending=False).reset_index(drop=True)

print('\n' + '='*65)
print(f'{"Modelo":<28} {"N":>6} {"MAE":>8} {"RMSE":>8} {"QWK":>8} {"F1":>8}')
print('-'*65)
for _, row in df_cot.iterrows():
    print(f'{row["modelo"]:<28} {int(row["n"]):>6} {row["mae"]:>8.4f} {row["rmse"]:>8.4f} {row["qwk"]:>8.4f} {row["f1"]:>8.4f}')
print('='*65)

df_cot.to_csv('resultados_exp4v2_final.csv', index=False)

cores = ['#4C72B0','#DD8452','#55A868','#C44E52']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Experimento 4 v2 — CoT + Optuna\nModelos Locais', fontsize=14, fontweight='bold')
for ax, (metrica, coluna) in zip(axes.flatten(),
        [('MAE','mae'),('RMSE','rmse'),('QWK','qwk'),('F1','f1')]):
    barras = ax.bar(df_cot['modelo'], df_cot[coluna], color=cores[:len(df_cot)], edgecolor='white')
    for b, v in zip(barras, df_cot[coluna]):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9)
    ax.set_title(metrica, fontsize=12, fontweight='bold')
    ax.set_xticklabels(df_cot['modelo'], rotation=15, ha='right', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('grafico_exp4v2_final.png', dpi=150, bbox_inches='tight')
plt.show()
print('Exp4 concluido! CSVs ck4_*.csv prontos para o Exp5.')

# EXPERIMENTO 5 — Instruction Tuning com Optuna

Usa os feedbacks gerados pelo Exp4 (ck4_*.csv) como gabarito de treinamento.
Os modelos sao fine-tunados via LoRA com Optuna otimizando os hiperparametros.

In [ ]:
# Celula - Construir dados de instruction tuning a partir dos CSVs do Exp4

def construir_par_it(redacao, score, c1, c2, c3, c4, c5, feedback):
    notas_json = (
        '{"C1": ' + str(c1) +
        ', "C2": ' + str(c2) +
        ', "C3": ' + str(c3) +
        ', "C4": ' + str(c4) +
        ', "C5": ' + str(c5) +
        ', "Nota_Total": ' + str(score) + '}'
    )
    instrucao = (
        'Voce e um avaliador oficial de redacoes do ENEM.\n'
        'Avalie a redacao fornecendo notas e feedback construtivo.\n'
        'Escala: 0, 40, 80, 120, 160 ou 200 por competencia.\n'
        'C1(Norma Culta) C2(Tema/Estrutura) C3(Argumentacao) C4(Coesao) C5(Intervencao)\n\n'
        'REDACAO:\n' + redacao
    )
    resposta = 'NOTAS:\n' + notas_json + '\n\nFEEDBACK:\n' + feedback
    return {'text': instrucao + '\n' + resposta}


# Carrega feedbacks dos CSVs do Exp4
dados_it = []
total_fb = 0

for nome_csv in ['ck4_llama.csv', 'ck4_mistral.csv', 'ck4_qwen.csv', 'ck4_gemma.csv']:
    if not os.path.exists(nome_csv):
        print(f'ATENCAO: {nome_csv} nao encontrado — pulando.')
        continue
    df_ck4 = pd.read_csv(nome_csv)
    df_ck4 = df_ck4[df_ck4['feedback'].notna()]
    df_ck4 = df_ck4[df_ck4['feedback'] != 'Feedback nao disponivel']
    df_ck4 = df_ck4[df_ck4['feedback'].str.len() > 20]

    for _, row_fb in df_ck4.iterrows():
        matches = df_teste[df_teste.index == row_fb['index_redacao']]
        if len(matches) > 0:
            rr = matches.iloc[0]
            dados_it.append(construir_par_it(
                rr['essay_limpo'], int(rr['score']),
                int(rr['c1']), int(rr['c2']), int(rr['c3']),
                int(rr['c4']), int(rr['c5']),
                str(row_fb['feedback'])
            ))
            total_fb += 1

# Complementa com feedbacks sinteticos do treino
for _, row in df_treino.head(300).iterrows():
    fb_sint = (
        'Nota total ' + str(row['score']) + '/1000. '
        'C1=' + str(row['c1']) + ' C2=' + str(row['c2']) +
        ' C3=' + str(row['c3']) + ' C4=' + str(row['c4']) +
        ' C5=' + str(row['c5']) + '.'
    )
    dados_it.append(construir_par_it(
        row['essay_limpo'], int(row['score']),
        int(row['c1']), int(row['c2']), int(row['c3']),
        int(row['c4']), int(row['c5']),
        fb_sint
    ))

print(f'Feedbacks do Exp4: {total_fb}')
print(f'Feedbacks sinteticos: 300')
print(f'Total pares IT: {len(dados_it)}')

In [ ]:
# Celula - Optuna e runner para Instruction Tuning

def objetivo_optuna_it(trial, nome_modelo):
    lr        = trial.suggest_float('lr', 1e-5, 5e-4, log=True)
    lora_rank = trial.suggest_categorical('lora_rank', [8, 16, 32])
    batch     = trial.suggest_categorical('batch', [1, 2])
    epochs    = trial.suggest_int('epochs', 1, 3)

    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=lora_rank, lora_alpha=lora_rank * 2,
        lora_dropout=0.05, bias='none',
        target_modules=['q_proj', 'v_proj']
    )

    tok, m = carregar_modelo(nome_modelo, para_treino=True)
    m = get_peft_model(m, lora_cfg)

    ds = Dataset.from_list(dados_it[:150]).map(
        lambda x: tok(x['text'], truncation=True, max_length=1024, padding='max_length'),
        remove_columns=['text']
    )
    args = TrainingArguments(
        output_dir=f'./opt_it_{trial.number}',
        num_train_epochs=epochs,
        per_device_train_batch_size=batch,
        gradient_accumulation_steps=max(1, 4 // batch),
        learning_rate=lr, bf16=True,
        logging_steps=50, save_strategy='no',
        report_to='none', warmup_steps=5
    )
    SFTTrainer(model=m, train_dataset=ds, args=args, processing_class=tok).train()

    m.eval()
    yp, yt = [], []
    for _, row in df_teste.head(30).iterrows():
        try:
            resp  = inf_local_it(tok, m, row['essay_limpo'])
            notas = extrair(resp)
            if notas and 'Nota_Total' in notas:
                yp.append(notas['Nota_Total'])
                yt.append(row['score'])
        except: pass

    liberar(m, tok)
    if len(yp) < 5: raise optuna.TrialPruned()
    def disc(n): return np.clip(np.round(np.array(n) / 40).astype(int), 0, 25)
    try:    return cohen_kappa_score(disc(yt), disc(yp), weights='quadratic')
    except: return -1.0


def rodar_optuna_it(nome_modelo, nome_curto, n_trials=4):
    print(f'Optuna IT: otimizando {nome_curto} ({n_trials} trials)...')
    study = optuna.create_study(
        direction='maximize',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=3)
    )
    study.optimize(lambda t: objetivo_optuna_it(t, nome_modelo), n_trials=n_trials, catch=(Exception,))
    print(f'Melhores params {nome_curto}: {study.best_params}')
    return study.best_params


def rodar_it(nome_modelo, nome_curto, params):
    lr        = params.get('lr', 2e-4)
    lora_rank = params.get('lora_rank', 16)
    batch     = params.get('batch', 2)
    epochs    = params.get('epochs', 3)

    print(f'\nInstruction Tuning {nome_curto} — lr={lr:.2e} rank={lora_rank} batch={batch} epochs={epochs}')

    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=lora_rank, lora_alpha=lora_rank * 2,
        lora_dropout=0.05, bias='none',
        target_modules=['q_proj', 'v_proj']
    )

    tok, m = carregar_modelo(nome_modelo, para_treino=True)
    m = get_peft_model(m, lora_cfg)
    m.print_trainable_parameters()

    ds = Dataset.from_list(dados_it).map(
        lambda x: tok(x['text'], truncation=True, max_length=2048, padding='max_length'),
        remove_columns=['text']
    )
    args = TrainingArguments(
        output_dir=f'./it_{nome_curto}',
        num_train_epochs=epochs,
        per_device_train_batch_size=batch,
        gradient_accumulation_steps=max(1, 4 // batch),
        learning_rate=lr, bf16=True,
        logging_steps=100, save_strategy='no',
        report_to='none', warmup_steps=20
    )
    SFTTrainer(model=m, train_dataset=ds, args=args, processing_class=tok).train()
    print(f'Instruction Tuning de {nome_curto} concluido!')

    m.eval()
    ckpt_csv  = f'ck5_{nome_curto}.csv'
    if os.path.exists(ckpt_csv):
        df_ck     = pd.read_csv(ckpt_csv)
        ja_feitos = set(df_ck['index_redacao'].tolist())
        print(f'Checkpoint: {len(ja_feitos)} ja avaliadas')
    else:
        df_ck     = pd.DataFrame(columns=['index_redacao', 'score', 'pred_total', 'feedback'])
        ja_feitos = set()

    pendentes = df_teste[~df_teste.index.isin(ja_feitos)]
    print(f'Pendentes: {len(pendentes)}/{len(df_teste)}')

    novos = []
    for idx, (i, row) in enumerate(pendentes.iterrows()):
        try:
            resp  = inf_local_it(tok, m, row['essay_limpo'])
            notas = extrair(resp)
            pred  = notas['Nota_Total'] if notas else None
            fb    = extrair_feedback(resp)
            novos.append({'index_redacao': i, 'score': row['score'], 'pred_total': pred, 'feedback': fb})
        except:
            novos.append({'index_redacao': i, 'score': row['score'], 'pred_total': None, 'feedback': None})

        if (idx + 1) % 50 == 0:
            df_ck = pd.concat([df_ck, pd.DataFrame(novos)], ignore_index=True)
            df_ck.to_csv(ckpt_csv, index=False)
            novos = []
            print(f'  Checkpoint: {idx + 1 + len(ja_feitos)}/{len(df_teste)}')

    if novos:
        df_ck = pd.concat([df_ck, pd.DataFrame(novos)], ignore_index=True)
        df_ck.to_csv(ckpt_csv, index=False)

    try:
        api   = HfApi()
        repo  = 'YurinhoMatsumoto/enem-' + nome_curto + '-it-exp5v2'
        pasta = './modelo_it_' + nome_curto
        m.save_pretrained(pasta)
        tok.save_pretrained(pasta)
        api.create_repo(repo_id=repo, token=HF_TOKEN, private=True, exist_ok=True)
        api.upload_folder(folder_path=pasta, repo_id=repo, token=HF_TOKEN)
        print(f'Modelo salvo no HF Hub: {repo}')
    except Exception as e:
        print(f'Erro ao salvar no HF Hub: {e}')

    liberar(m, tok)

    df_v = df_ck.dropna(subset=['pred_total'])
    if len(df_v) > 0:
        return calcular_metricas(df_v['score'].tolist(), df_v['pred_total'].tolist(), nome_curto + ' (IT)')
    return None


print('Optuna IT e runner carregados!')

## Llama 3.1 8B — Instruction Tuning

In [ ]:
params_llama_it = rodar_optuna_it('meta-llama/Llama-3.1-8B-Instruct', 'llama', n_trials=4)
res_llama_it = rodar_it('meta-llama/Llama-3.1-8B-Instruct', 'llama', params_llama_it)

## Mistral 7B — Instruction Tuning

In [ ]:
params_mistral_it = rodar_optuna_it('mistralai/Mistral-7B-Instruct-v0.3', 'mistral', n_trials=4)
res_mistral_it = rodar_it('mistralai/Mistral-7B-Instruct-v0.3', 'mistral', params_mistral_it)

## Qwen 2.5 7B — Instruction Tuning

In [ ]:
params_qwen_it = rodar_optuna_it('Qwen/Qwen2.5-7B-Instruct', 'qwen', n_trials=4)
res_qwen_it = rodar_it('Qwen/Qwen2.5-7B-Instruct', 'qwen', params_qwen_it)

## Gemma 2 9B — Instruction Tuning

In [ ]:
params_gemma_it = rodar_optuna_it('google/gemma-2-9b-it', 'gemma', n_trials=4)
res_gemma_it = rodar_it('google/gemma-2-9b-it', 'gemma', params_gemma_it)

## Resultados Exp5

In [ ]:
# Resultados Exp5
todos_it = [res_llama_it, res_mistral_it, res_qwen_it, res_gemma_it]
df_it = pd.DataFrame([r for r in todos_it if r is not None])
df_it = df_it.sort_values('qwk', ascending=False).reset_index(drop=True)

print('\n' + '='*65)
print(f'{"Modelo":<28} {"N":>6} {"MAE":>8} {"RMSE":>8} {"QWK":>8} {"F1":>8}')
print('-'*65)
for _, row in df_it.iterrows():
    print(f'{row["modelo"]:<28} {int(row["n"]):>6} {row["mae"]:>8.4f} {row["rmse"]:>8.4f} {row["qwk"]:>8.4f} {row["f1"]:>8.4f}')
print('='*65)

df_it.to_csv('resultados_exp5v2_final.csv', index=False)

cores = ['#4C72B0','#DD8452','#55A868','#C44E52']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Experimento 5 v2 — Instruction Tuning + Optuna\nModelos Locais', fontsize=14, fontweight='bold')
for ax, (metrica, coluna) in zip(axes.flatten(),
        [('MAE','mae'),('RMSE','rmse'),('QWK','qwk'),('F1','f1')]):
    barras = ax.bar(df_it['modelo'], df_it[coluna], color=cores[:len(df_it)], edgecolor='white')
    for b, v in zip(barras, df_it[coluna]):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9)
    ax.set_title(metrica, fontsize=12, fontweight='bold')
    ax.set_xticklabels(df_it['modelo'], rotation=15, ha='right', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('grafico_exp5v2_final.png', dpi=150, bbox_inches='tight')
plt.show()
print('Exp5 concluido!')

## Exemplos de Feedback — Exp4 e Exp5

In [ ]:
# Exemplos de feedback gerado
print('=== FEEDBACKS EXP4 (CoT) ===')
for var, nome in [('llama','Llama'),('mistral','Mistral'),('qwen','Qwen'),('gemma','Gemma')]:
    arq = f'ck4_{var}.csv'
    if not os.path.exists(arq): continue
    df_ex = pd.read_csv(arq).dropna(subset=['feedback'])
    df_ex = df_ex[df_ex['feedback'].str.len() > 20]
    if len(df_ex) == 0: continue
    print(f'\n--- {nome} (CoT) ---')
    print(f'Score humano : {df_ex["score"].iloc[0]}')
    print(f'Score modelo : {df_ex["pred_total"].iloc[0]}')
    print(f'Feedback     :\n{str(df_ex["feedback"].iloc[0])[:300]}...')

print('\n=== FEEDBACKS EXP5 (Instruction Tuning) ===')
for var, nome in [('llama','Llama'),('mistral','Mistral'),('qwen','Qwen'),('gemma','Gemma')]:
    arq = f'ck5_{var}.csv'
    if not os.path.exists(arq): continue
    df_ex = pd.read_csv(arq).dropna(subset=['feedback'])
    df_ex = df_ex[df_ex['feedback'].str.len() > 20]
    if len(df_ex) == 0: continue
    print(f'\n--- {nome} (IT) ---')
    print(f'Score humano : {df_ex["score"].iloc[0]}')
    print(f'Score modelo : {df_ex["pred_total"].iloc[0]}')
    print(f'Feedback     :\n{str(df_ex["feedback"].iloc[0])[:300]}...')